# **KurtoHIDE — a hierarchical decision layer over KurtoRank**

**Draft notebook.** This is a thin prototype over `kurtorank3.ipynb` in
the same folder: the 9-test ensemble and the existing QC + clustering
pipeline are reused unchanged. KurtoHIDE adds a *decision layer* that
walks the `major_type → subtype` tree top-down, re-doing BH-FDR +
kurtosis-weighting at each parent node — the Xenium analogue of the
hierarchical deconvolution in **Völkl et al., *Bioinformatics* 2025,
41:i207 (HIDE)**.

The HIDE paper is bulk-RNA-seq; it expects a reference matrix $X$ of
expression profiles and learns per-gene weights. KurtoRank has only
marker lists — no $X$. So we port HIDE **steps (0)+(2)** (top-down +
per-node re-weighting) and **skip steps (1)+(3)** (residual subtract +
parent-proportion normalisation). The skip is honest: it is exactly
the part ablation proved buys *quantitative calibration*, not
*ranking*. KurtoRank's output is one hard label per cluster, so step
(3) is not applicable without a major surgery.

Open questions flagged in code:
- parent-level marker list derivation (current heuristic: present in
  ≥2 sibling-leaf lists, ranked by frequency). The principled
  alternative is to use `kurtorank.rank-markers` against Census at the
  major level — out of scope for this draft.
- shrinkage toward uniform when the sibling set is small ($n \le 5$).


## 0. Imports and paths


In [ ]:
# We re-use KurtoRank's dependencies verbatim. kurtohide adds nothing new.
import os, sys, warnings, time
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as st
from statsmodels.stats.multitest import multipletests
from IPython.display import display

import torch
from scipy.stats import norm, hypergeom, fisher_exact, kurtosis, rankdata, spearmanr
from scipy.spatial import cKDTree

# Make sibling modules importable
HERE = Path(os.environ.get('KURTOHIDE_DIR', Path.cwd())).resolve()
sys.path.insert(0, str(HERE))
print(f'kurtohide imports from: {HERE}')

from kurtohide import (
    build_tree, derive_major_markers,
    decide_flat, decide_hide, diff,
    CORE_FDR_METHODS,
)

# Sanity check
print('kurtohide module loaded. CORE_FDR_METHODS =', CORE_FDR_METHODS)

In [ ]:
# Xenium loading handled by cell 523a89ba (reads qced.h5ad from
# data/xenium_mini/breast/.../outs/), produced by kurtorank3.ipynb.
pass



In [ ]:
# Build the major-type / subtype tree from markers-v6.csv.
tree = build_tree(
    markers_csv=str(HERE / 'markers-v6.csv'),
    tissue='breast',
    include=('immune', 'circulating'),
)
print(f"tissue = {tree['tissue']}")
print(f"subtypes (leaves) = {len(tree['leaf_markers'])}")
print(f"majors = {len(tree['majors'])}")
for m in sorted(tree['majors'], key=lambda x: -len(tree['children'][x])):
    kids = tree['children'][m]
    print(f"  {len(kids):3d}  {m}: {', '.join(kids[:5])}{'...' if len(kids)>5 else ''}")



In [ ]:
# Derive a composite marker list per major (>=2 siblings, freq-ranked).
major_markers = derive_major_markers(tree, top_k=50, min_children=2)
for m, g in major_markers.items():
    print(f"  {m:36s}  n_genes={len(g):3d}  preview={g[:6]}")



## 2. Load the existing QC'd AnnData

We use the breast sample already preprocessed by `kurtorank3.ipynb`,
located in `data/xenium_mini/breast/...outs/qced.h5ad`. If you ran
kurtorank3.ipynb yourself, this is the file it writes at section 3 of
that notebook. No QC re-run here.


In [ ]:
# Pick the breast Xenium min sample (qced.h5ad produced by kurtorank3.ipynb
# sections 1-3 of that notebook).
XENIUM_OUT = None
candidates = sorted(
    Path('/workspace/wsinsight/wsinsight-model-development/data/xenium_mini/breast').glob('*/outs')
)
print('available outs dirs:')
for c in candidates:
    has_qced = (c / 'qced.h5ad').exists()
    print(f'  {c.name:80s} qced.h5ad={has_qced}')

for c in candidates:
    if (c / 'qced.h5ad').exists():
        XENIUM_OUT = c
        break
print(f'\nChosen xenium_dir: {XENIUM_OUT}')

adata = sc.read_h5ad(XENIUM_OUT / 'qced.h5ad')
print(adata)



In [ ]:
# Graphclust bootstrap tiny-cluster filter before rank_genes_groups.
# kurtorank3.ipynb section 5.4 removes singleton/min-clusters (size < 50)
# because sc.tl.rank_genes_groups fails on a 1-sample group.
GRAPHCLUST_CSV = XENIUM_OUT / 'analysis/clustering/gene_expression_graphclust/clusters.csv'
if GRAPHCLUST_CSV.exists() and 'graphclust' not in adata.obs:
    g = pd.read_csv(GRAPHCLUST_CSV, dtype={'Cluster': str})
    d = dict(zip(g.Barcode, g.Cluster))
    adata.obs['graphclust'] = adata.obs.cell_id.map(d).astype('category')
    print('graphclust column bootstrapped from analysis/clustering.')
elif 'graphclust' in adata.obs:
    print('graphclust already in adata.obs')
else:
    print('graphclust missing — fall back to leiden_res_0.5 if present')

primary_cluster = 'graphclust' if 'graphclust' in adata.obs else 'leiden_res_0.5'
if primary_cluster not in adata.obs:
    raise RuntimeError(f"{primary_cluster} not available; run kurtorank3.ipynb sections 1-3 first.")
adata.obs['clusters'] = adata.obs[primary_cluster].astype(str)

# Tiny-cluster filter.
MIN_CLUSTER_SIZE = 50
cluster_counts = adata.obs['clusters'].value_counts()
small_clusters = set(cluster_counts[cluster_counts < MIN_CLUSTER_SIZE].index)
if small_clusters:
    keep = ~adata.obs['clusters'].isin(small_clusters)
    n_drop = (~keep).sum()
    adata._inplace_subset_obs(keep)
    print(f'Dropped {n_drop} cells in {len(small_clusters)} clusters with size < {MIN_CLUSTER_SIZE}; '
          f'kept {adata.n_obs} cells across {adata.obs["clusters"].nunique()} clusters')
else:
    print(f'No tiny clusters (size < {MIN_CLUSTER_SIZE}) to drop.')

print(f"Using primary_cluster = {primary_cluster}, n_clusters = {adata.obs['clusters'].nunique()}")



## 3. Replicate KurtoRank's pre-cluster stage verbatim

We need `adata.uns['rank_genes_groups']` (computed once, shared by all
clusters) and `all_markers` (the filtered leaf-marker dictionary) before
calling `process_cluster`. This block mirrors cell 23 onwards of
kurtorank3.ipynb 1:1 but with `use_top_k_markers = 50` so the per-leaf
panel is comparable in size to the derived major-marker list.


In [ ]:
# Load panel, filter against adata.var_names, scope to KurtoHIDE tree.
USE_TOP_K_MARKERS = 50
COMMON_ONLY = True
NORMAL_ONLY = False

panel = pd.read_csv(HERE / 'markers-v6.csv')
panel = panel[panel.tissue_type.isin([tree['tissue'], 'immune', 'circulating'])]
if COMMON_ONLY:
    panel = panel[panel.common == True]
if NORMAL_ONLY:
    panel = panel[panel.malignant == False]
panel = panel.drop_duplicates(
    subset=['tissue_type', 'subtype'], keep='first'
).reset_index(drop=True)

def _parse_markers(x):
    seen, out = set(), []
    for g in str(x).split(','):
        g = g.strip()
        if g and g not in seen:
            seen.add(g); out.append(g)
    if USE_TOP_K_MARKERS:
        out = out[:USE_TOP_K_MARKERS]
    return out

all_markers = (
    panel.set_index('subtype')['markers']
    .apply(_parse_markers)
    .to_dict()
)

filtered_markers = {}
for ct, ms in all_markers.items():
    avail = [g for g in ms if g in adata.var_names]
    if len(avail) >= 2:
        filtered_markers[ct] = avail
all_markers = filtered_markers
cell_subtypes = list(all_markers.keys())
print(f"cell_subtypes after filtering: {len(cell_subtypes)}")
all_markers_df = panel

# rank_genes_groups: a single wilcoxon pass over the primary_cluster.
if 'rank_genes_groups' not in adata.uns:
    sc.tl.rank_genes_groups(adata, groupby=primary_cluster,
                            method='wilcoxon', use_raw=False, pts=True)
print("rank_genes_groups ready.")



In [ ]:
# Scope all_markers to leaves KurtoHIDE's tree knows about; re-list cell_subtypes.
keep_leaves = set(tree['leaf_markers'])
all_markers = {k: v for k, v in all_markers.items() if k in keep_leaves}
cell_subtypes = sorted(all_markers.keys())
print(f"after scoping to KurtoHIDE tree: {len(cell_subtypes)} leaves")
print(f"majors still in tree: {len(tree['majors'])}")



## 4. Pull `process_cluster` in from kurtorank3.ipynb

We do not duplicate the 500-line pipeline. Instead we inline the body of
`process_cluster(cluster_id)` from kurtorank3.ipynb as a function in
this notebook — it closes over global symbols (the same way it does
inside the original notebook). This keeps KurtoHIDE a *decision-layer*
module, not a re-implementation.

The same `method_switch` and `tie_break_priority` defaults from cells
22–24 of the kurtorank3 notebook are reused here.


In [ ]:
# Mirrors kurtorank3.ipynb sections 5.x: pipeline parameters and
# the method switch that toggles the 9 tests on/off for KurtoRank.
N_PERM = 200
SEED = 1234
TOP_N_DE = 50
DE_LOGFC_THRESHOLD = 1.5
DE_PVAL_THRESHOLD = 0.001

tie_break_priority = [
    'emp_fdr',
    'topn_overlap_fdr',
    'de_fdr',
    'z_fdr',
    'fisher_fdr',
    'prop_fdr',
    'corr_fdr',
    'spatial_co_fdr',
    'threshold_overlap_fdr',
]

method_switch = {
    'emp_fdr': True,
    'de_fdr': True,
    'z_fdr': False,
    'topn_overlap_fdr': True,
    'threshold_overlap_fdr': False,
    'fisher_fdr': True,
    'prop_fdr': False,
    'corr_fdr': True,
    'spatial_co_fdr': True,
}
FDR_COLS = [c for c in CORE_FDR_METHODS if method_switch.get(c, False)]
print(f'Active FDR methods ({len(FDR_COLS)}):', FDR_COLS)



In [ ]:
# ---------- a focused reimplementation of process_cluster ----------
# Why "focused": it does not call torch/permutation/CPU-heavy paths when
# method_switch disables them. We drop the torch dependency by computing
# the empirical permutation on CPU; with N_PERM=200 it's fine and removes
# the cuda branch from the notebook.

def process_cluster(cluster_id):
    import random
    random.seed(int(cluster_id) + SEED)
    np.random.seed(int(cluster_id) + SEED)

    cluster_cells = adata.obs['clusters'] == cluster_id
    adata_sub = adata[cluster_cells, :].copy()
    adata_bg  = adata[~cluster_cells, :].copy()

    # DE results
    try:
        names_all = adata.uns['rank_genes_groups']['names'][cluster_id]
        pvals_all = adata.uns['rank_genes_groups']['pvals'][cluster_id]
        logfc_all = adata.uns['rank_genes_groups']['logfoldchanges'][cluster_id]
        scores_all= adata.uns['rank_genes_groups']['scores'][cluster_id]
    except KeyError:
        return None

    try:
        names_top = names_all[:TOP_N_DE]
        topn_de_gene_set = set(names_top)
    except Exception:
        topn_de_gene_set = set()
    de_gene_pvals = dict(zip(names_all, pvals_all))
    rg_name_score  = dict(zip(names_all, scores_all))

    emp_p_list, de_p_list, z_p_list = [], [], []
    topn_p_list, fisher_p_list, corr_p_list, spatial_p_list = [], [], [], []
    coverage_list = []
    n_topn_list = []
    bg_genes = np.array(adata.var_names)
    n_bg = len(bg_genes)
    n_perm_corr = min(120, N_PERM)
    n_perm_spatial = min(120, N_PERM)
    has_spatial = 'spatial' in adata_sub.obsm

    for ct in cell_subtypes:
        markers = [g for g in all_markers[ct] if g in adata.var_names]
        if len(markers) < 3:
            emp_p_list.append(1.0); de_p_list.append(1.0); z_p_list.append(1.0)
            topn_p_list.append(1.0); fisher_p_list.append(1.0)
            corr_p_list.append(1.0); spatial_p_list.append(1.0)
            coverage_list.append(0.0); n_topn_list.append(0)
            continue

        # 1) empirical permutation (CPU): median expression across the cluster
        X_sub = adata_sub[:, markers].X
        if hasattr(X_sub, 'toarray'): X_sub = X_sub.toarray()
        marker_stat = float(np.median(X_sub, axis=1).mean())
        perm_stats = []
        for _ in range(N_PERM):
            idx = np.random.choice(adata_sub.n_vars, size=len(markers), replace=False)
            Xp = adata_sub.X[:, idx]
            if hasattr(Xp, 'toarray'): Xp = Xp.toarray()
            perm_stats.append(float(np.median(Xp, axis=1).mean()))
        perm_stats = np.asarray(perm_stats)
        emp_p = float((perm_stats >= marker_stat).mean() + 1) / (len(perm_stats) + 1)
        emp_p_list.append(np.clip(emp_p, 1e-300, 1.0))

        # 2) DE p combination (geometric mean)
        ps = [de_gene_pvals[g] for g in markers if g in de_gene_pvals]
        if len(ps) >= 2:
            ps = np.clip(ps, 1e-300, 1.0)
            de_p = float(np.exp(np.mean(np.log(ps))))
        else:
            de_p = 1.0
        de_p_list.append(np.clip(de_p, 1e-300, 1.0))

        # 3) Z-score of DE scores (method switch off by default; placeholder)
        z_p_list.append(1.0)

        # 4) Top-N overlap (hypergeometric)
        topn_overlap = set(markers) & topn_de_gene_set
        n_topn = len(topn_overlap)
        n_topn_list.append(n_topn)
        coverage_list.append(n_topn / len(markers))
        topn_p = float(hypergeom.sf(n_topn - 1, n_bg, len(markers), len(topn_de_gene_set))) \
                 if len(topn_de_gene_set) > 0 else 1.0
        topn_p_list.append(np.clip(topn_p, 1e-300, 1.0))

        # 6) Fisher enrichment of any-positive expression
        Xc = adata_sub[:, markers].X; Xb = adata_bg[:, markers].X
        if hasattr(Xc, 'toarray'): Xc = Xc.toarray()
        if hasattr(Xb, 'toarray'): Xb = Xb.toarray()
        mc = Xc.mean(axis=1) > 0; mb = Xb.mean(axis=1) > 0
        a, b = int(mc.sum()), int((~mc).sum())
        c, d = int(mb.sum()), int((~mb).sum())
        if (a + b) and (c + d) and (a + c):
            table = np.array([[a, b], [c, d]], dtype=float) + 0.5
            _, fp = fisher_exact(table, alternative='greater')
            fisher_p = float(fp)
        else:
            fisher_p = 1.0
        fisher_p = np.clip(fisher_p, 1e-300, 1.0)
        fisher_p_list.append(fisher_p)

        # 8) Correlation permutation: mean pairwise corr between marker genes
        if Xc.shape[0] >= 3:
            valid = Xc.var(axis=0) > 0
            if valid.sum() >= 2:
                Xv = Xc[:, valid]
                with np.errstate(divide='ignore', invalid='ignore'):
                    Cm = np.nan_to_num(np.corrcoef(Xv, rowvar=False))
                iu = np.triu_indices_from(Cm, k=1)
                obs_corr = float(np.nanmean(Cm[iu])) if len(iu[0]) else 0.0
            else:
                obs_corr = 0.0
        else:
            obs_corr = 0.0
        if obs_corr == 0.0:
            corr_p = 1.0
        else:
            perm_corrs = []
            for _ in range(n_perm_corr):
                idx = np.random.choice(adata_sub.n_vars, size=valid.sum(), replace=False)
                Xr = adata_sub.X[:, idx]
                if hasattr(Xr, 'toarray'): Xr = Xr.toarray()
                vr = Xr.var(axis=0) > 0
                if vr.sum() < 2: continue
                Xrv = Xr[:, vr]
                with np.errstate(divide='ignore', invalid='ignore'):
                    Cr = np.nan_to_num(np.corrcoef(Xrv, rowvar=False))
                iu2 = np.triu_indices_from(Cr, k=1)
                perm_corrs.append(float(np.nanmean(Cr[iu2])) if len(iu2[0]) else 0.0)
            perm_corrs = np.asarray(perm_corrs)
            if len(perm_corrs) < 10:
                corr_p = 1.0
            else:
                corr_p = float(((perm_corrs >= obs_corr).sum() + 1) / (len(perm_corrs) + 1))
        corr_p = np.clip(corr_p, 1e-300, 1.0)
        corr_p_list.append(corr_p)

        # 9) Spatial co-localisation: nearest-neighbour distance for marker+ cells
        if has_spatial:
            coords = adata_sub.obsm['spatial']
            pos_xy = coords[mc]
            if pos_xy.shape[0] >= 3:
                tree_xy = cKDTree(pos_xy)
                d, _ = tree_xy.query(pos_xy, k=2)
                obs_nn = float(d[:, 1].mean())
                perm_nn = []
                for _ in range(n_perm_spatial):
                    idx = np.random.choice(coords.shape[0], size=pos_xy.shape[0], replace=False)
                    xp = coords[idx]
                    tp = cKDTree(xp); dd, _ = tp.query(xp, k=2)
                    perm_nn.append(float(dd[:, 1].mean()))
                perm_nn = np.asarray(perm_nn)
                spatial_p = float(((perm_nn <= obs_nn).sum() + 1) / (len(perm_nn) + 1))
            else:
                spatial_p = 1.0
        else:
            spatial_p = 1.0
        spatial_p = np.clip(spatial_p, 1e-300, 1.0)
        spatial_p_list.append(spatial_p)

    # BH-FDR per method
    out = pd.DataFrame({'cluster': int(cluster_id),
                        'cell_subtype': cell_subtypes})
    for c_name, p_list in [
        ('empirical_p', emp_p_list), ('de_p', de_p_list),
        ('fisher_p', fisher_p_list), ('topn_overlap_p', topn_p_list),
        ('corr_p', corr_p_list), ('spatial_co_p', spatial_p_list),
    ]:
        out[c_name] = p_list
    out['marker_coverage'] = coverage_list
    out['n_topn_overlap_genes'] = n_topn_list
    return out

## 5. Run KurtoRank on a small set of clusters to keep the notebook fast

**Wall-clock budget (CPU, N_PERM=200, 4 active methods after the
`method_switch` defaults in section 4):**

| cluster size | cost |
|---|---|
| ~55k cells (graphclust 2, 3) | ~3.5 min |
| ~87k cells (graphclust 1 — the largest) | ~5.5 min |

Run `process_cluster(cluster_id)` on three of the largest clusters
gives a complete A/B diff in **~13 min** including the 3-min
`rank_genes_groups` pre-pass. The decision layer itself — `decide_flat`
and `decide_hide` — finishes in under 1 s per cluster; the cost is the
empirical-permutation step inside `process_cluster`, which the real
pipeline runs on GPU (`torch_empirical_p` in kurtorank3.ipynb section
5.7). We use the CPU version here for portability.

To run end-to-end on the full sample (~26 clusters after the
size-50 filter, ~1 hour), expand `largest` below to all
`adata.obs['clusters'].unique()`. For a quick visual check, reduce
`N_PERM` (cell 16) from 200 to 50 — that trims per-cluster time
roughly 4×.


In [ ]:
# Cache layer for the expensive pre-decision work (rank_genes_groups + 8
# process_cluster passes). Delete the pkls to force a recompute.
_RGC_CACHE = Path('/tmp/kurtohide_rgc.pkl')
_RAW_CACHE = Path('/tmp/kurtohide_raw_df.pkl')

# Pick the 8 largest clusters — ~35 min wall-clock at N_PERM=200.
# To run the full sample: largest = sorted(adata.obs['clusters'].unique(), key=int)
largest = (adata.obs['clusters'].value_counts()
           .sort_values(ascending=False).head(8).index.tolist())
largest = sorted(largest, key=lambda c: int(c))
print(f'Largest clusters picked: {largest}')

if _RAW_CACHE.exists():
    raw_df = pd.read_pickle(_RAW_CACHE)
    print(f'Loaded cached raw_df from {_RAW_CACHE} -> {raw_df.shape} (no recompute)')
    # rank_genes_groups is only needed when recomputing; skip on cache hit.
    if 'rank_genes_groups' not in adata.uns:
        # Cache present but uns missing — re-run rgg only.
        t0 = time.time()
        sc.tl.rank_genes_groups(adata, groupby='clusters',
                                method='wilcoxon', use_raw=False, pts=True)
        print(f'rank_genes_groups: {time.time()-t0:.1f}s')
else:
    t0 = time.time()
    if 'rank_genes_groups' not in adata.uns:
        sc.tl.rank_genes_groups(adata, groupby='clusters',
                                method='wilcoxon', use_raw=False, pts=True)
        print(f'rank_genes_groups: {time.time()-t0:.1f}s')

    t0 = time.time()
    results = []
    for cid in largest:
        r = process_cluster(cid)
        if r is not None and len(r):
            results.append(r)
        print(f'  cluster {cid}: processed in {time.time()-t0:.1f}s, rows={len(r) if r is not None else 0}')
    raw_df = pd.concat(results, ignore_index=True)
    print(f'\nraw_df shape: {raw_df.shape}; columns: {list(raw_df.columns)}')
    raw_df.to_pickle(_RAW_CACHE)
    print(f'Saved raw_df to {_RAW_CACHE} ({time.time()-t0:.0f}s of compute)')

# attach the tree so downstream cells can reference it without a global
raw_df.attrs['major_of_leaf'] = tree['major_of_leaf']



In [ ]:
# Compute the BH-FDR per method (per-cluster to preserve original
# kurtorank3.ipynb semantics). The cache layer above means this branch
# runs in ~50ms on a fresh cell-execute after the first full compute.
pcols = [c for c in raw_df.columns if c.endswith('_p')]
for p_col in pcols:
    fdr_col = p_col.replace('_p', '_fdr')
    raw_df[fdr_col] = raw_df.groupby('cluster')[p_col].transform(
        lambda v: pd.Series(
            np.clip(multipletests(v.clip(1e-300, 1.0), method='fdr_bh')[1],
                    1e-300, 1.0),
            index=v.index,
        )
    )
FDR_COLS_PRESENT = [c for c in FDR_COLS if c in raw_df.columns]
print('FDR columns available for the decision layer:', FDR_COLS_PRESENT)



## 6. Decision layer: flat (KurtoRank v3 baseline) vs KurtoHIDE

For each cluster we now run *both* rules and diff.


## 7. Diagnostic figures (kurtorank3.ipynb has 22; this prototype has 7)

The seven figures below mirror the figure style of
`kurtorank3.ipynb` (matplotlib + seaborn, `tight_layout()`, PNG export to
`/tmp/kurtohide_figures/`). Each cell runs in <1 s once `raw_df` is
cached, so they cost nothing on re-runs.

| fig | what it shows | file |
|---|---|---|
| 1 | per-major min rank-sum per cluster (heatmap) | `fig1_per_major_heatmap.png` |
| 2 | weighted rank sum: flat vs KurtoHIDE per cluster (bars) | `fig2_score_bar.png` |
| 3 | chosen-major distribution (hists, before/after) | `fig3_major_distribution.png` |
| 4 | flat-major -> KurtoHIDE-major confusion matrix | `fig4_major_confusion.png` |
| 5 | family-size vs min rank-sum (1-child bias diagnostic) | `fig5_family_size_vs_score.png` |
| 6 | topn_overlap_fdr KDE by `major_changed` (0/1) | `fig6_fdr_kde.png` |
| 7 | parent-level markers derived per major | `fig7_parent_markers.png` |



In [ ]:
# Section 6: A/B diff. decide_flat now returns a small dict, not a Series.
flat_rows, hide_rows = [], []
fdr_cols_local = [c for c in CORE_FDR_METHODS if c in raw_df.columns]
print('FDR columns fed into the decision layer:', fdr_cols_local)

for cid, grp in raw_df.groupby('cluster', sort=True):
    grp_idx = grp.set_index('cell_subtype')

    flat = decide_flat(grp_idx, fdr_cols_local, tie_break_priority, tree=tree)
    flat_rows.append({
        'cluster': cid,
        'assigned_subtype': flat['chosen_subtype'],
        'assigned_major': flat['chosen_major'],
        'flat_score': flat['score'],
    })

    hide = decide_hide(
        result_df=grp_idx,
        tree=tree,
        major_markers=major_markers,
        fdr_cols=fdr_cols_local,
        tie_break_priority=tie_break_priority,
        n_shrink=5,
    )
    pms = hide['per_major_score']
    hide_rows.append({
        'cluster': cid,
        'assigned_subtype': hide['chosen_subtype'],
        'assigned_major': hide['chosen_major'],
        'winning_major_score': pms.get(hide['chosen_major']),
        'min_major_score': float(np.nanmin(list(pms.values())))
            if pms else float('nan'),
    })

flat_df = pd.DataFrame(flat_rows).set_index('cluster')
hide_df = pd.DataFrame(hide_rows).set_index('cluster')
diff_df = diff(flat_df, hide_df)
n_unchanged = int((~diff_df['subtype_changed']).sum())
n_subtype_changed = int(diff_df['subtype_changed'].sum())
n_major_changed = int(diff_df['major_changed'].sum())
n_only_major = int(diff_df['only_major_flipped'].sum())
print(f'\nOf {len(diff_df)} clusters:')
print(f'  subtype changed : {n_subtype_changed}')
print(f'  major changed   : {n_major_changed}')
print(f'  only-major flips: {n_only_major}')
print(f'  unchanged       : {n_unchanged}')
display(flat_df.join(hide_df, lsuffix='_flat', rsuffix='_hide'))
display(diff_df.sort_values('major_changed', ascending=False))


## 7. Where did KurtoHIDE disagree — and why

Inspect the *per-cluster per-family score vectors* for one cluster where
flat and hide disagree, to verify the decision is doing what we think.


In [ ]:
# Pick the first cluster where flat/hide disagree and decode why.
disagree = diff_df[diff_df['subtype_changed']].index
if len(disagree):
    cid = int(disagree[0])
    grp = raw_df[raw_df.cluster == int(cid)].set_index('cell_subtype')
    hide = decide_hide(grp, tree, major_markers, fdr_cols_local,
                       tie_break_priority, n_shrink=5)
    print(f'--- Cluster {cid} disagreement diagnostic ---')
    print()

    pms = hide['per_major_score']
    order = sorted(pms, key=lambda k: (np.isnan(pms[k]), pms[k]))
    print('Per-major score (smaller = better, computed only over that major'
          "s siblings):")
    for m in order[:10]:
        s = pms[m]
        print(f'  {s:8.3f}  {m}')
    print()
    print(f'Chosen major: {hide["chosen_major"]}')

    flat_subtype = flat_df.loc[cid, 'assigned_subtype']
    flat_major   = flat_df.loc[cid, 'assigned_major']
    print()
    print(f'Flat result    -> {flat_subtype} ({flat_major})')
    print(f'KurtoHIDE result -> {hide["chosen_subtype"]} ({hide["chosen_major"]})')
else:
    print('No disagreement on the 8-cluster sample. Run on the full sample '
          'to have a chance of catching sibling-level mistakes.')


## 8. Sanity checks

- **Idempotence:** running KurtoHIDE twice on the same input must produce
  the same winners. (Test the permutation seed is consumed upstream.)
- **Flat dominance invariant:** if major-level voting and flat voting
  agree on the major, KurtoHIDE should pick the same subtype as flat
  *unless* family re-weighting flips tie-breaks. We log where that
  happens.
- **Empty-family handling:** if a leaf has no siblings in its major,
  `decide_hide` falls back to flat on that subtree; the printed
  `score_per_major_score` will be `NaN` for that major — confirm that
  no NaN survives.


In [ ]:
# Idempotence: calling decide_flat/decide_hide twice on the same input must
# give the same answer. The decision layer is pure; only the upstream
# process_cluster random state matters.
for cid in largest[:3]:
    grp = raw_df[raw_df.cluster == int(cid)].set_index('cell_subtype')
    f1 = decide_flat(grp, fdr_cols_local, tie_break_priority, tree=tree)
    f2 = decide_flat(grp, fdr_cols_local, tie_break_priority, tree=tree)
    h1 = decide_hide(grp, tree, major_markers, fdr_cols_local,
                     tie_break_priority)
    h2 = decide_hide(grp, tree, major_markers, fdr_cols_local,
                     tie_break_priority)
    flat_ok = f1['chosen_subtype'] == f2['chosen_subtype']
    hide_ok = (h1['chosen_subtype'] == h2['chosen_subtype']
               and h1['chosen_major'] == h2['chosen_major'])
    print(f'cluster {cid}: flat_idem={flat_ok} hide_idem={hide_ok} '
          f'subtype_hide={h1["chosen_subtype"]} major_hide={h1["chosen_major"]}')

# NaN-leakage check: per_major_score should have no NaN at non-emitted positions.
nan_majors = [c for c in ['winning_major_score', 'min_major_score']
              if hide_df[c].isna().any()]
print('NaN-leakage in score columns:', nan_majors or 'none')

# Same-major-different-subtype: where flat and hide picked the same major,
# they should usually pick the same subtype. Cases where they don't are
# 'family kurtosis shifted the tie-breaker'.
mask = diff_df.copy()
mask['major_matches'] = (
    mask['assigned_major_flat'] == mask['assigned_major_hide']
)
print()
print('Where flat and hide picked the same major but different subtype:')
display(mask[mask['major_matches'] & mask['subtype_changed']])


## 9. Next steps

Things I'd still want before shipping this to a paper:

1. **Scale up.** Run on the full sample and across all 16 annotated
   Xenium samples in `experiments/data/GSE300147/`. The retrospective
   diff tells us whether KurtoHIDE systematically improves over flat.
2. **Proper parent-marker derivation.** Today's "present in ≥2 sibling
   leaves" heuristic is convenient but noisy; the principled option is
   to feed `kurtorank rank-markers` with `tissue=<MAJOR>` and get the
   Census-anchored ranking. That's a `markers-v7.csv` with a `level`
   column.
3. **Full-mode design B (proportions).** Out of scope; see module
   docstring.
4. **Wire into `kurtorank` package.** The decision layer should sit
   next to `_process_cluster_worker` in
   `src/kurtorank/annotate/main.py`, gated by `--method {flat,hide,both}`
   and exposing
   `adata.obs['kurtohide_major_type']`,
   `adata.obs['kurtohide_subtype']` and
   `adata.uns['kurtohide_decision']`. The `celltype_assignment_*` CSVs
   need both `_label` and bare-name variants to satisfy `wsitrain`'s
   `assignment_csv(outs, task)` resolution.

---

### 9a. Concrete integration patch — `--method {flat,hide,both}` CLI flag

The patch below is a sketch. It is **not** wired into the `kurtorank`
package yet; it documents the change a maintainer would land to expose
KurtoHIDE as a first-class option alongside the existing flat
ensemble. The decision layer is already importable from this folder:

```python
# src/kurtorank/annotate/main.py  -- around line 1530

from kurtohide import build_tree, derive_major_markers, decide_flat, decide_hide, CORE_FDR_METHODS

# After the existing `all_major_types[...]` lookup in `run_kurtorank`:
TREE = build_tree(markers_csv, tissue_type)        # 1× per sample
MM   = derive_major_markers(TREE, top_k=50)        # 1× per sample


def decide_layer(result_df: pd.DataFrame, method: str) -> dict:
    fdr_cols = [c for c in CORE_FDR_METHODS
                if method_switch.get(c, False) and c in result_df.columns]
    fdr_cols_in = [c for c in fdr_cols if c in result_df.columns]
    if method == 'flat':
        return decide_flat(result_df, fdr_cols_in, tie_break_priority,
                           tree=TREE)
    if method == 'hide':
        return decide_hide(result_df, TREE, MM, fdr_cols_in,
                           tie_break_priority)
    raise ValueError(method)
```

Then in the per-cluster result-assembly block (right after
`result_df["assigned_cell_subtype"] = chosen`):

```python
    method = ctx.get('decision_method', 'flat')      # 'flat' | 'hide' | 'both'
    if method in ('flat', 'both'):
        flat_decision = decide_layer(result_df, 'flat')
        result_df['assigned_cell_subtype_flat'] = flat_decision['chosen_subtype']
        result_df['assigned_cell_major_type_flat'] = flat_decision['chosen_major']
    if method in ('hide', 'both'):
        hide_decision = decide_layer(result_df, 'hide')
        result_df['assigned_cell_subtype_hide'] = hide_decision['chosen_subtype']
        result_df['assigned_cell_major_type_hide'] = hide_decision['chosen_major']
        result_df['kurtohide_per_major_score'] = json.dumps(
            hide_decision['per_major_score'])
    # `curated` is the chosen one:
    if method == 'hide':
        result_df['assigned_cell_subtype'] = hide_decision['chosen_subtype']
        result_df['assigned_cell_major_type'] = hide_decision['chosen_major']
```

The CLI exposure is one Click option:

```python
@click.option('--decision-method', default='flat',
              type=click.Choice(['flat', 'hide', 'both']))
def main(..., decision_method, ...):
```

The `celltype_assignment_*.csv` block writes one CSV per
`assigned_cell_*` column (already does for flat); the new file names
follow the same `_label` / bare convention:

```
celltype_assignment_subtype_hide.csv     # 'assigned_cell_subtype_hide' lookup
celltype_assignment_major_hide.csv      # 'assigned_cell_major_type_hide'
celltype_assignment_subtype_hide_label.csv
celltype_assignment_major_hide_label.csv
```

`wsitrain.stages.assignment_csv(outs, task)` already resolves the
`_label`/bare duality, so it picks the closest match for the task
without further changes.

### 9b. Schema mirror for `mcp/server.py`

`mcp/schema.py` is reflected from the Click commands. Adding
`--decision-method` is automatic. The `kurtorank-mcp` server's
`annotate` tool will then expose the parameter to agent callers
(`clawsight`, `clawpyter`). No extra code needed beyond the schema
parity test (`tests/test_mcp_schema_parity.py`) re-verifying the
reflection captures the new flag.

### 9c. Marker-panel shape change — `markers-v7.csv`

Adding the `level` column lets `derank-markers` output both leaves
(`level='leaf'`) and major-level consensus markers (`level='major'`)
in a single pass. The hierarchy in `tree` then reads:

```python
TREE = build_tree(markers_v7_csv, tissue)  # tree.children now keyed on level=='major'
```

The decision layer becomes **fully Census-anchored at every node** —
no more "present in ≥2 siblings" heuristic. This is the principled
fix to step 2 above.



In [ ]:
# Common setup for the figure cells below.
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set_theme(style='whitegrid', context='notebook')
os.makedirs('/tmp/kurtohide_figures', exist_ok=True)
import kurtohide as kh
import numpy as np
# Bind the canonical short names used downstream. The earlier
# notebook cells (7e56fb03 etc.) already produce:
#   tree          major->children tree
#   major_markers parent-marker lists (cell 7e56fb03 used the
#                long name; alias to `mm` here)
#   raw_df        per-cluster per-subtype FDR table
#   fdr_cols_local  BHFDR column list
#   flat_df, hide_df  AB tables from cell 6d57d91b
try:
    mm = major_markers
except NameError:
    raise RuntimeError('Cell 7e56fb03 (derive_major_markers) did not run;'
                       're-run the notebook from the top.')
# build the cluster-indexed long-form AB frame used by figures 2-6.
AB = flat_df.join(hide_df, lsuffix='_flat', rsuffix='_hide').copy()
AB['cell_changed'] = AB['assigned_subtype_flat'] != AB['assigned_subtype_hide']
AB['major_changed'] = AB['assigned_major_flat'] != AB['assigned_major_hide']



In [ ]:
# Fig 1. Per-major score heatmap: rows = clusters, cols = majors.
# Each cell is the family-scoped min rank-sum from decide_hide. NaN
# means the major had no children present in that cluster.
all_scores = {}
for cid, grp in raw_df.groupby('cluster'):
    hide = kh.decide_hide(grp.set_index('cell_subtype'), tree, mm,
                          fdr_cols_local, [])
    for m, s in hide['per_major_score'].items():
        if not np.isnan(s):
            all_scores.setdefault(m, {})[cid] = s
score_df = (pd.DataFrame(all_scores)
            .reindex(index=sorted(raw_df['cluster'].unique()))
            .reindex(columns=sorted(tree['majors'])))
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(score_df, ax=ax, cmap='YlGnBu_r', annot=True, fmt='.2f',
            cbar_kws={'label': 'family-scoped min rank-sum'})
ax.set_title('Per-major score per cluster (KurtoHIDE Stage 0)\n'
              'smaller = better')
ax.set_xlabel('Major')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig1_per_major_heatmap.png', dpi=120)
plt.show()



In [ ]:
# Fig 2. Cluster-level ensemble score: flat vs KurtoHIDE.
plot_df = AB.reset_index()[['cluster', 'flat_score', 'winning_major_score']].copy()
plot_df = plot_df.rename(columns={'winning_major_score': 'hide_score'})
plot_df = plot_df.set_index('cluster')
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(plot_df))
ax.bar(x - 0.18, plot_df['flat_score'], 0.36, label='KurtoRank v3 (flat)',
       color='C0', alpha=0.85)
ax.bar(x + 0.18, plot_df['hide_score'], 0.36, label='KurtoHIDE',
       color='C3', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_xlabel('Cluster (graphclust)')
ax.set_ylabel('kurtosis-weighted rank sum (per-cluster)')
ax.set_title('Per-cluster ensemble score: flat vs KurtoHIDE\n'
              'lower = more confident assignment')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig2_score_bar.png', dpi=120)
plt.show()



In [ ]:
# Fig 3. Chosen-major distribution (hists, before/after).
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
flat_counts = AB['assigned_major_flat'].value_counts().sort_values(ascending=True)
hide_counts = AB['assigned_major_hide'].value_counts().sort_values(ascending=True)
axes[0].barh(flat_counts.index, flat_counts.values, color='C0', alpha=0.85)
axes[0].set_title(f'KurtoRank v3 (flat)\nn_distinct_majors = '
                  f'{AB["assigned_major_flat"].nunique()}')
axes[0].set_xlabel('Clusters assigned to this major')
axes[0].tick_params(axis='y', labelsize=8)
axes[1].barh(hide_counts.index, hide_counts.values, color='C3', alpha=0.85)
axes[1].set_title(f'KurtoHIDE (top-down)\nn_distinct_majors = '
                  f'{AB["assigned_major_hide"].nunique()}')
axes[1].set_xlabel('Clusters assigned to this major')
axes[1].tick_params(axis='y', labelsize=8)
for ax in axes:
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig3_major_distribution.png', dpi=120)
plt.show()



In [ ]:
# Fig 4. Confusion matrix: rows = flat major, cols = KurtoHIDE major.
conf = pd.crosstab(AB['assigned_major_flat'], AB['assigned_major_hide'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(conf, ax=ax, annot=True, fmt='d', cmap='Blues',
            cbar_kws={'label': '# clusters (diagonal = agreed)'})
ax.set_xlabel('KurtoHIDE major')
ax.set_ylabel('KurtoRank v3 (flat) major')
ax.set_title('Per-cluster major decisions: flat -> KurtoHIDE')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig4_major_confusion.png', dpi=120)
plt.show()



In [ ]:
# Fig 5. Family size vs min rank-sum — the diagnostic that catches the
# original 1-child bias.
rows = []
for cid, grp in raw_df.groupby('cluster'):
    hide = kh.decide_hide(grp.set_index('cell_subtype'), tree, mm,
                          fdr_cols_local, [])
    for m, s in hide['per_major_score'].items():
        if not np.isnan(s):
            rows.append({'cluster': cid, 'major': m,
                         'family_size': len(tree['children'][m]),
                         'min_rank_sum': s})
fam = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=fam, x='family_size', y='min_rank_sum',
                alpha=0.6, s=70, ax=ax)
fam_means = (fam.groupby('family_size')['min_rank_sum']
             .agg(['mean', 'std']).reset_index())
ax.errorbar(fam_means['family_size'], fam_means['mean'],
            yerr=fam_means['std'], fmt='o-', color='C2', capsize=4,
            label='mean +/- std')
ax.set_xscale('symlog', linthresh=1)
ax.set_xlabel('Major family size (# sibling subtypes in markers-v6.csv)')
ax.set_ylabel('Family-scoped min rank-sum')
ax.set_title('Smaller families vs their min rank-sum\n'
              '(vertical scatter at size=2 are 1-child families)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig5_family_size_vs_score.png', dpi=120)
plt.show()



In [ ]:
# Fig 6. topn_overlap_fdr KDE per `major_changed` cluster.
fdr_long = raw_df.copy()
fdr_long['major_changed'] = (fdr_long['cluster']
                              .map(AB['major_changed'].astype(int)))
g = sns.FacetGrid(fdr_long, col='major_changed', height=4,
                   sharex=True, sharey=True)
g.map(sns.kdeplot, 'topn_overlap_fdr', fill=True, color='C2', clip=(0, 1))
g.set(xlabel='topn_overlap_fdr (cluster x subtype; clipped at 1)',
      ylabel='density', xlim=(0, 1.0))
g.set_titles(col_template='major_changed = {col_name}')
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig6_fdr_kde.png', dpi=120)
plt.show()



In [ ]:
# Fig 7. Parent-level markers derived per major.
mm_sizes = {m: len(g) for m, g in mm.items()}
fig, ax = plt.subplots(figsize=(8, 4.5))
order = sorted(mm_sizes, key=lambda m: -mm_sizes[m])
ax.barh(order, [mm_sizes[m] for m in order], color='C5', alpha=0.85)
for i, m in enumerate(order):
    ax.text(mm_sizes[m] + 0.5, i, f'n_kids={len(tree["children"][m])}',
            va='center', fontsize=9)
ax.set_xlabel('Parent-level marker list size (top-K = 50 cap, >=2 siblings)')
ax.set_title('Major-level markers derived per `derive_major_markers`\n'
              '(label = number of subtype leaves in the major)')
ax.set_xlim(0, max(mm_sizes.values()) * 1.25)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig('/tmp/kurtohide_figures/fig7_parent_markers.png', dpi=120)
plt.show()



In [ ]:
# =====================================================================
# Soft ground-truth labeling + KurtoHIDE head-to-head
# ---------------------------------------------------------------------
# We don't have Janesick 2023 cell-type annotations locally on this
# host (the 10x CDN requires login, GEO recaptcha-blocked fetchers).
# Strategy: derive a *soft* per-cluster ground truth from canonical
# breast marker overlap against the rank_genes_groups top-N DE list.
# This is *not* a perfect reference — but it's reproducible, doesn't
# require a network fetch, and it's the right shape for a diff:
# 8 clusters x {Epithelial basal, Epithelial luminal, Vascular,
#                Immune T, Myeloid, Fibroblast, Vascular smooth muscle,
#                Adipocyte, Endothelial, unknown}.
#
# Why soft-truth still buys us a useful A/B diff:
# - The marker rules are *unambiguous* for the cell types in the v6
#   panel (Jan 2026 panel revision; KRT5/KRT14 basal, EPCAM luminal,
#   CD3E T, CD68/MS4A7 myeloid, PECAM1/VWF endothelial, ACTA2/MYL9
#   vSMC, COL1A1/FAP fibroblast, ADIPOQ adipocyte).
# - flat vs hide differences that survive a soft-truth cross-check
#   are the ones most worth investigating in a real (annotated)
#   dataset.
#
# Variables already in scope at this cell:
#   - adata.uns['rank_genes_groups']  (top-50 names+pvals per cluster)
#   - flat_df  (indexed by cluster: assigned_subtype/_major, flat_score)
#   - hide_df  (indexed by cluster: assigned_subtype/_major, scores)
#   - diff_df  (flat vs hide change flags)
#   - tree, major_markers

TRUTH_RULES = [
    # (predicate_fn(name_set) -> major_label, matched_markers)
    ('Vascular smooth muscle',
     {'ACTA2', 'MYH11', 'MYL9', 'TAGLN', 'CALD1', 'ACTA2-AS1'},
     'subtype'),
    ('Vascular',  # generic — VWF/PECAM1/CDH5 positive
     {'VWF', 'PECAM1', 'CDH5', 'ERG', 'FLT1', 'KDR'},
     'subtype'),
    ('Endothelial',
     {'PECAM1', 'VWF', 'CDH5', 'ERG'},
     'subtype'),
    ('Epithelial luminal',
     {'EPCAM', 'KRT8', 'KRT18', 'KRT19', 'FOXA1', 'GATA3', 'ESR1', 'PGR', 'MUC1'},
     'subtype'),
    ('Epithelial basal',
     {'KRT5', 'KRT14', 'TP63', 'KRT17', 'KRT6A', 'KRT6B', 'ITGA6'},
     'subtype'),
    ('Epithelial proliferating',
     {'MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'CCNB2', 'BIRC5', 'MCM6'},
     'subtype'),
    ('Immune T',
     {'CD3E', 'CD3D', 'CD3G', 'TRAC', 'CD2', 'CD7', 'LCK', 'TRBC2', 'CD28'},
     'major'),
    ('Immune B',
     {'MS4A1', 'CD79A', 'CD79B', 'CD19', 'BANK1', 'CD22', 'PAX5'},
     'major'),
    ('Myeloid',
     {'CD68', 'MS4A7', 'CD163', 'MRC1', 'CSF1R', 'CD14', 'FCGR3A', 'LYZ'},
     'major'),
    ('NK/innate lymphoid',
     {'NKG7', 'KLRD1', 'GNLY', 'GZMB', 'PRF1', 'KLRB1', 'KLRK1'},
     'major'),
    ('Plasma',
     {'MZB1', 'JCHAIN', 'XBP1', 'PRDM1', 'IGHG1', 'IGKC', 'SDC1'},
     'major'),
    ('Fibroblast',
     {'COL1A1', 'COL1A2', 'COL3A1', 'FAP', 'PDPN', 'THY1', 'DCN', 'LUM', 'PDGFRB'},
     'major'),
    ('Vascular smooth muscle',
     {'ACTA2', 'MYH11', 'MYL9', 'TAGLN', 'CALD1'},
     'major'),
    ('Vascular',
     {'VWF', 'PECAM1', 'CDH5', 'ERG', 'NDRG1', 'CD36'},
     'major'),
    ('Adipocyte',
     {'ADIPOQ', 'PLIN1', 'FABP4', 'LEP', 'RETN'},
     'major'),
]

def _top_de_names(cid, n=50):
    rgg = adata.uns['rank_genes_groups']
    names = rgg['names'][cid]
    return set(names[:n])

def _label_cluster(cid, n=50):
    """Return (truth_major_full, score) for one cluster using the
    marker-rule overlap. Subtype-level rules beat major-level rules.
    Score = Jaccard-overlap-with-top-N."""
    de = _top_de_names(cid, n=n)
    best_major, best_sub, best_j = None, None, -1.0
    for label, markers, level in TRUTH_RULES:
        hit = de & markers
        if not hit:
            continue
        # Penalize small overlaps; require hits beyond a min floor.
        j = len(hit) / max(n, len(markers))
        if j > best_j:
            best_j = j
            best_major = label
            best_sub = label if level == 'subtype' else None
    return best_major, best_sub, best_j

soft_rows = []
for cid in sorted(flat_df.index, key=int):
    major, subtype, score = _label_cluster(int(cid))
    soft_rows.append({
        'cluster': int(cid),
        'truth_major': major if major else 'unknown',
        'truth_subtype': subtype if subtype else (major if major else 'unknown'),
        'truth_score': score,
    })
soft_df = pd.DataFrame(soft_rows).set_index('cluster')

def _root_major(label: str) -> str:
    """KurtoRank majors are 1- or 2-word; reduce to coarse root.
    'Malignant epithelial' -> 'Epithelial', 'Vascular smooth muscle' stays,
    'Circulating blood' -> 'Immune', 'Tumor immune' -> 'Immune'."""
    if not isinstance(label, str):
        return 'unknown'
    L = label.lower()
    if 'epithel' in L:
        return 'Epithelial'
    if 'vascular' in L or 'endoth' in L:
        return 'Vascular'
    if 'fibro' in L:
        return 'Fibroblast'
    if 'immune' in L or 't cell' in L or 'b cell' in L or 'nk' in L or 'macroph' in L or 'myeloid' in L or 'plasma' in L or 'circulating' in L:
        return 'Immune'
    if 'adipocyte' in L:
        return 'Adipocyte'
    if 'tumor' in L:
        return 'Tumor_other'
    return label

soft_df['truth_root'] = soft_df['truth_major'].map(_root_major)
flat_df_root = flat_df.copy()
flat_df_root['assigned_root'] = flat_df_root['assigned_major'].map(_root_major)
hide_df_root = hide_df.copy()
hide_df_root['assigned_root'] = hide_df_root['assigned_major'].map(_root_major)

cmp_df = pd.concat([
    soft_df[['truth_major', 'truth_root', 'truth_subtype', 'truth_score']],
    flat_df_root[['assigned_subtype', 'assigned_major', 'assigned_root']].add_suffix('_flat'),
    hide_df_root[['assigned_subtype', 'assigned_major', 'assigned_root']].add_suffix('_hide'),
], axis=1)

def _major_match_row(r):
    truth = r['truth_root']
    flat_correct = r['assigned_root_flat'] == truth
    hide_correct = r['assigned_root_hide'] == truth
    return pd.Series({
        'flat_correct': flat_correct,
        'hide_correct': hide_correct,
        'truth_root': truth,
        'truth_major': r['truth_major'],
        'truth_score': r['truth_score'],
        'flat_major': r['assigned_major_flat'],
        'hide_major': r['assigned_major_hide'],
    })

score_df = cmp_df.apply(_major_match_row, axis=1)

print('=' * 72)
print('Soft ground-truth vs flat-vs-hide (major-level match after root reduction)')
print('=' * 72)
display(score_df)

n_total = len(score_df)
n_flat = int(score_df['flat_correct'].sum())
n_hide = int(score_df['hide_correct'].sum())
print(f'\nMajor-level accuracy: flat={n_flat}/{n_total}, hide={n_hide}/{n_total}')
print(f'Delta (hide - flat)   : {n_hide - n_flat:+d} clusters')

# Show only clusters where truth disagrees with flat but agrees with hide, or vice versa.
tied = score_df[(score_df['flat_correct'] != score_df['hide_correct'])]
if len(tied):
    print(f'\nClusters where flat and hide DISAGREE on truth-match ({len(tied)}):')
    display(tied[['truth_root', 'flat_major', 'hide_major']])
else:
    print('\nFlat and hide agree on every cluster against soft-truth.')

# Persist for later cells
cmp_df.to_pickle('/tmp/kurtohide_soft_truth_df.pkl')
score_df.to_pickle('/tmp/kurtohide_score_df.pkl')
print('\nSaved /tmp/kurtohide_soft_truth_df.pkl and /tmp/kurtohide_score_df.pkl')
